In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

## $\text{\textcolor{green}{Day-long Horizon Solver w/ Curtailment}}$


In [ ]:
# Daily solver
def solve_battery_day(day_df, minimum_self_sufficiency, charge_eff, discharge_eff, dT, C_rate, load_profile):
    day_df = day_df.sort_index()

    PV_gen = day_df["PV_total"].to_numpy()
    Wind_gen = day_df["Wind_total"].to_numpy()
    n = len(day_df)

    Pload_np = load_profile
    
    if len(Pload_np) != n:
        return {"status": "bad_input: load length mismatch"}

    if not np.isfinite(Pload_np).all():
        return {"status": "bad_input: non-finite load"}

    if not np.isfinite(PV_gen).all():
        return {"status": "bad_input: non-finite PV"}

    if not np.isfinite(Wind_gen).all():
        return {"status": "bad_input: non-finite wind"}

    M = 1.1 * max(np.max(Pload_np), np.max(PV_gen + Wind_gen))

    if not np.isfinite(M) or M <= 0:
        return {"status": "bad_input: invalid M"}

    # Big-M from generation and load to relate the max power and energy capacities to them
    M = 1.1 * max(np.max(Pload_np), np.max(PV_gen + Wind_gen))

    # Constants
    Pload = cp.Constant(Pload_np)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)

    # Variables
    P_capacity = cp.Variable(nonneg=True)
    E_capacity = cp.Variable(nonneg=True)

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)
    z = cp.Variable(n, boolean=True)

    Pgrid_buy = cp.Variable(n, nonneg=True)
    SOC = cp.Variable(n)
    Pcurtail = cp.Variable(n, nonneg= True)

    constraints = [
        P_capacity <= C_rate * E_capacity,

        Pcharge <= P_capacity,
        Pdischarge <= P_capacity,

        Pcharge <= M * z,
        Pdischarge <= M * (1 - z),

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Curtailment can never exceed available generation
        Pcurtail <= Pgen_PV + Pgen_W,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge + Pcurtail,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[n - 1] >= 0.5 * E_capacity,

        #cap on import from the grid based on difference of self-sufficiency
        
        cp.sum(Pgrid_buy) * dT <= (1 - minimum_self_sufficiency) * np.sum(Pload_np) * dT
    ]

    # Extras for curtailment -- penalty to avoid spilling and tracker of spillage
    spilled_energy = cp.sum(Pcurtail) * dT
    eps_curt = 10

    problem = cp.Problem(cp.Minimize(E_capacity + eps_curt * spilled_energy), constraints)

    try:
        problem.solve(solver=cp.CBC, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            return {
            "status": problem.status,
            "E_capacity_kWh": np.nan,
            "P_capacity_kW": np.nan,
            "spilled_energy_kWh": np.nan,
            "grid_and_curtail_simultaneous": np.nan
            }

        both_active = np.any(
            (Pgrid_buy.value > 1e-6) & (Pcurtail.value > 1e-6)
        )

        return {
            "status": problem.status,
            "E_capacity_kWh": E_capacity.value,
            "P_capacity_kW": P_capacity.value,
            "spilled_energy_kWh": np.sum(Pcurtail.value) * dT,
            "grid_and_curtail_simultaneous": both_active
        }

    except Exception as e:
        return {
            "status": f"solver_error: {type(e).__name__}: {str(e)}",
            "E_capacity_kWh": np.nan,
            "P_capacity_kW": np.nan,
            "spilled_energy_kWh": np.nan,
            "grid_and_curtail_simultaneous": np.nan
        }

## $\text{\textcolor{green}{Day-long Horizon for Battery Sizing w/ Variable Load \& Curtailment}}$


$\text{\textcolor{green}{Load}}$

In [ ]:
df_load = pd.read_csv(
    ROOT / "data/archive/archive/building_consumption.csv",
    parse_dates=["timestamp"]
)

filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14 07:30:00") &
    (df_load["timestamp"] <= "2022-04-10 23:30:00")
]

total_load = (
    filtered
    .groupby("timestamp")["consumption"]
    .sum()
    .reset_index(name="total_consumption")
)

total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean()

load_scale = 0.004
total_load_30min["total_consumption"] *= load_scale

total_load_30min["date"] = total_load_30min.index.date
total_load_30min["month_day"] = total_load_30min.index.strftime("%m-%d")

load_counts = total_load_30min.groupby("date")["total_consumption"].count()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[
    total_load_30min["date"].isin(load_complete_days)
].copy()

load_day_profiles = []

for date, day_df in load_days_df.groupby("date"):
    day_df = day_df.sort_index()

    load_day_profiles.append({
        "load_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "Load_profile": day_df["total_consumption"].to_numpy()
    })

load_profiles_df = pd.DataFrame(load_day_profiles)
print(load_profiles_df.columns)

When scaling down by 94.4 %, 132 days return feasible optimizations and you have a battery size for each of the self-sufficiency targets set out -- 50% through 100% 

$\text{\textcolor{green}{Generation}}$

In [ ]:
DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"

df_gen = pd.read_csv(
    DATA_DIR,
    delimiter=",",
    decimal=".",
    parse_dates=["ts"],
    index_col="ts"
)

df_agg_gen = df_gen.copy()

df_agg_gen["PV_total"] = df_agg_gen[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg_gen["Wind_total"] = df_agg_gen[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg_gen.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)

df_agg_gen["date"] = df_agg_gen.index.date
df_agg_gen["month_day"] = df_agg_gen.index.strftime("%m-%d")

counts_per_day = df_agg_gen.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index

df_agg_gen = df_agg_gen[df_agg_gen["date"].isin(complete_days)].copy()

gen_day_profiles = []

for date, day_df in df_agg_gen.groupby("date"):
    day_df = day_df.sort_index()

    pv_profile = day_df["PV_total"].to_numpy()
    wind_profile = day_df["Wind_total"].to_numpy()
    renewable_profile = pv_profile + wind_profile

    gen_day_profiles.append({
        "gen_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "PV_profile": pv_profile,
        "Wind_profile": wind_profile,
        "Renewable_profile": renewable_profile
    })

gen_profiles_df = pd.DataFrame(gen_day_profiles)
#print(gen_day_profiles)

$\text{\textcolor{purple}{I'm only getting 6 results for an optimal battery size across all self-sufficiencies, i.e. a single pair of generation and load days is able to meet any and all of the self-sufficiency goals}}$

I have 330 full-length generation days and 332 full-length load days

My generation doesn't seem to be able to satisfy the load. If this load is realistic for a scenario like Risø, then we would need to size up!

$\text{\textcolor{green}{Joining Load and Generation Data \& Visualization of Data}}$

In [ ]:
paired_day_profiles = load_profiles_df.merge(gen_profiles_df, on="month_day", how="inner")

paired_day_profiles = (paired_day_profiles.sort_values("month_day").reset_index(drop=True))

paired_day_profiles.columns

#print("Load days:", len(load_profiles_df))
#print("Generation days:", len(gen_profiles_df))
#print("Paired days:", len(paired_day_profiles))

#print(paired_day_profiles[["load_date", "gen_date", "month_day"]].head())

#print(paired_day_profiles["Load_profile"].apply(len).value_counts())

#print(paired_day_profiles["Renewable_profile"].apply(len).value_counts())


In [ ]:
#Visualize load

time_labels = pd.date_range("00:00", periods=48, freq="30min").strftime("%H:%M")

plt.figure(figsize=(10, 6))

for profile in paired_day_profiles["Load_profile"]:
    plt.plot(time_labels, profile, alpha=0.25)

plt.xticks(time_labels[::4], rotation=45)
plt.xlabel("Time of day")
plt.ylabel("Load [kW]")
plt.title("Daily load profiles")
plt.grid(True)
plt.show()

# Visualize generation 

plt.figure(figsize=(10, 6))

for profile in paired_day_profiles["Renewable_profile"]:
    plt.plot(time_labels, profile, alpha=0.25)

plt.xticks(time_labels[::4], rotation=45)
plt.xlabel("Time of day")
plt.ylabel("Generation [kW]")
plt.title("Daily generation profiles")
plt.grid(True)
plt.show()


$\text{\textcolor{green}{Net Load and Self-Sufficiency Diagnostic}}$

In [ ]:
diagnostics = []
dT = 0.5  # still needed for cumulative behavior

for _, row in paired_day_profiles.iterrows():
    load_profile = row["Load_profile"]
    gen_profile = row["Renewable_profile"]

    net_load = load_profile - gen_profile
    cumulative = np.cumsum(net_load * dT)

    # --- Power-based metrics ---
    avg_load_kW = load_profile.mean()
    avg_gen_kW = gen_profile.mean()

    peak_load_kW = load_profile.max()
    peak_gen_kW = gen_profile.max()

    # Direct overlap in power (instantaneous matching)
    direct_used_profile = np.minimum(load_profile, gen_profile)
    avg_direct_used_kW = direct_used_profile.mean()

    # Mismatch metrics
    mean_abs_mismatch_kW = np.mean(np.abs(net_load))

    diagnostics.append({
        "load_date": row["load_date"],
        "gen_date": row["gen_date"],
        "month_day": row["month_day"],

        # --- Power comparison ---
        "avg_load_kW": avg_load_kW,
        "avg_gen_kW": avg_gen_kW,
        "avg_direct_used_kW": avg_direct_used_kW,

        "peak_load_kW": peak_load_kW,
        "peak_gen_kW": peak_gen_kW,

        "max_deficit_kW": np.maximum(net_load, 0).max(),
        "max_surplus_kW": np.maximum(-net_load, 0).max(),

        "mean_abs_mismatch_kW": mean_abs_mismatch_kW,

        # Still useful (links to storage need)
        "ideal_storage_swing_kWh": cumulative.max() - cumulative.min()
    })

diagnostics_df = pd.DataFrame(diagnostics)

diagnostics_df.describe()

$\text{\textcolor{green}{Daily Profile Optimizations For Battery Size Based on a Target Self-Sufficiency}}$

In [ ]:
targets = np.linspace(0.1, 1.0, 10)

results = []

for target in targets:
    for _, row in paired_day_profiles.iterrows():

        load_profile = row["Load_profile"]

        day_df = pd.DataFrame({
            "PV_total": row["PV_profile"],
            "Wind_total": row["Wind_profile"]
        })

        if len(day_df) != 48 or len(load_profile) != 48:
            continue

        out = solve_battery_day(
            day_df,
            minimum_self_sufficiency=target,
            charge_eff=0.95,
            discharge_eff=0.95,
            dT=0.5,
            C_rate=0.5,
            load_profile=load_profile
        )

        out["load_date"] = row["load_date"]
        out["gen_date"] = row["gen_date"]
        out["month_day"] = row["month_day"]
        out["target_self_sufficiency"] = target

        results.append(out)

optimization_results_df = pd.DataFrame(results)

print(optimization_results_df)
print(optimization_results_df["status"].value_counts(dropna=False))

if "grid_and_curtail_simultaneous" in optimization_results_df.columns:
    print(optimization_results_df["grid_and_curtail_simultaneous"].value_counts(dropna=False))

Result Nomenclature:

- optimal: model found a solution
- infeasible = constrainst are imporssible for that day/target, i.e. model was processed but no satisfactory solution was found
- Solver_error = something "broke" before attemptiong solution due to bad numbers/dimensions or numerical instability

In [ ]:
valid = optimization_results_df[
    optimization_results_df["status"] == "optimal"
].copy()

print("Total runs:", len(optimization_results_df))
print("Optimal runs:", len(valid))

summary = valid.groupby("target_self_sufficiency").agg(
    median_E=("E_capacity_kWh", "median"),
    p90_E=("E_capacity_kWh", lambda x: x.quantile(0.9)),
    max_E=("E_capacity_kWh", "max"),

    median_P=("P_capacity_kW", "median"),
    p90_P=("P_capacity_kW", lambda x: x.quantile(0.9)),
    max_P=("P_capacity_kW", "max"),

    median_spilled=("spilled_energy_kWh", "median"),
    p90_spilled=("spilled_energy_kWh", lambda x: x.quantile(0.9)),
    max_spilled=("spilled_energy_kWh", "max"),

    feasible_days=("E_capacity_kWh", "count")
).reset_index()

print(summary)

$\text{\textcolor{green}{Correlation Between Diagnostic and Optimization Results}}$

In [ ]:
# Merging diagnostics with optimization

analysis_df = optimization_results_df.merge(
    diagnostics_df,
    on="month_day",
    how="left"
)

#Storage energy vs required swing -- Expects a strong positive correlation, and the optimizer to be a "realistic version" of the ideal swing
target = 0.9
subset = analysis_df[analysis_df["target_self_sufficiency"] == target]

plt.figure(figsize=(6, 5))
plt.scatter(subset["ideal_storage_swing_kWh"],subset["E_capacity_kWh"])
plt.xlabel("Ideal storage swing (kWh)")
plt.ylabel("Optimized battery size (kWh)")
plt.title(f"Storage need vs optimizer result (target={target})")
plt.grid()
plt.show()

#Power mismatch vs battery power size --> P_capacity_kW should scale with extreme mismatch

plt.figure(figsize=(6, 5))
plt.scatter(subset["max_deficit_kW"],subset["P_capacity_kW"],label="Deficit")
plt.scatter(subset["max_surplus_kW"],subset["P_capacity_kW"],label="Surplus")
plt.xlabel("Power mismatch (kW)")
plt.ylabel("Battery power capacity (kW)")
plt.title(f"Power sizing vs mismatch (target={target})")
plt.legend()
plt.grid()
plt.show()

#Difficulty vs battery size --> shows that messier days require larger batteries

plt.figure(figsize=(6, 5))
plt.scatter(subset["mean_abs_mismatch_kW"],subset["E_capacity_kWh"])
plt.xlabel("Mean mismatch (kW)")
plt.ylabel("Battery energy (kWh)")
plt.title("Average mismatch vs storage size")
plt.grid()
plt.show()

### $\text{\textcolor{green}{Plot for Best Battery Size vs. Self-Sufficiency}}$

$\text{\textcolor{green}{Sizing According to Energy Capacity}}$

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(summary["self_sufficiency"] * 100, summary["median_E"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_E"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_E"], marker="o", label="Maximum")
plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Required energy capacity [kWh]")
plt.title("Battery energy size vs self-sufficiency target")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

$\text{\textcolor{green}{Sizing According to Power Capacity}}$

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(summary["self_sufficiency"] * 100, summary["median_P"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_P"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_P"], marker="o", label="Maximum")
plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Required power capacity [kW]")
plt.title("Battery power size vs self-sufficiency target")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### $\text{\textcolor{green}{Plot for Distribution of Battery Sizes}}$

$\text{\textcolor{green}{Energy Capacity Distribution}}$

In [ ]:
plt.figure(figsize=(9, 4))

valid_plot = valid.copy()
valid_plot["self_sufficiency_pct"] = (valid_plot["self_sufficiency"] * 100).round()
valid_plot.boxplot(column="E_capacity_kWh", by="self_sufficiency_pct")

plt.title("Distribution of daily energy capacity by self-sufficiency target")
plt.suptitle("")  # removes automatic pandas title

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Energy capacity [kWh]")

plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

$\text{\textcolor{green}{Power Capacity Distribution}}$

In [ ]:
plt.figure(figsize=(9, 4))

valid_plot = valid.copy()
valid_plot["self_sufficiency_pct"] = (valid_plot["self_sufficiency"] * 100).round()
valid_plot.boxplot(column="P_capacity_kW", by="self_sufficiency_pct")

plt.title("Distribution of daily power capacity by self-sufficiency target")
plt.suptitle("")

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Power capacity [kW]")

plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### $\text{\textcolor{green}{Plot for Spillage According to Self-Sufficiency}}$

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(summary["self_sufficiency"] * 100, summary["median_spilled"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_spilled"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_spilled"],marker="o", label="Maximum")

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Spilled renewable energy [kWh/day]")
plt.title("Renewable curtailment vs self-sufficiency target")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

Why can spillage still happen if the battery could, in theory, absorb that energy?

Exploration Studies:
- input power as a daily profile => you want to get a distribution of battery capacities
    - would this distribution also have to vary according to self-sufficiencies?
- limit P capacity in reference to C capacity => with C-rate or a weight called something arbitrary
    - different c-rates => compare battery technologies/types of lithium ion batteries in energy systems

Small size of battery could be due to:
- enough generation to not require battery
- lax constraint on self-sufficiency

Make sure that when you're charging your'e not discharding
    - maybe introduce boolean variable to prevent simultaneous operation

target function could be to minimize SOC * battery E capacity * 1.0 or just the E_capacity
you need to make the 

Meeting April 21st Notes
- Change data into daily profiles => you want to get a distribution of battery sizes depending on the profile of days
    *modify daily profile function from daily profile clustering + incorporate aggregation function.
- Start optimization with real data to have comparison in terms of accuracy
    - assume you have next-day's weather and power values
- Upon having comparison, you can incorporate clustering results
- optimization code should be wrapped in a function
    - input: 24-hour of real pv power and real wind
    - output: schedule for battery operation

- Load data may exist in Gaggle or we can simulate it from DTU's load data
    - doesn't overlap directly timewise => don't take average so you have a spikier profile